In [1]:
# Cell 1 â€” Install dependencies
!pip install scipy numpy matplotlib torch torchvision \
    torch-geometric umap-learn wandb networkx tqdm -q

ModuleNotFoundError: No module named 'google'

In [ ]:
# Cell 2 â€” Clone repo (re-clones every session; pulls latest if already exists)
import os
REPO_ROOT = '/content/antenna-gnn'
if not os.path.exists(REPO_ROOT):
    try:
        from google.colab import userdata
        github_token = userdata.get('GITHUB_TOKEN')
        !git clone https://{github_token}@github.com/asparagusD/antenna_gnn.git {REPO_ROOT}
    except userdata.SecretNotFoundError:
        print('GITHUB_TOKEN not found in Colab Secrets. Attempting public clone...')
        !git clone https://github.com/asparagusD/antenna_gnn.git {REPO_ROOT}
else:
    !git -C {REPO_ROOT} pull --quiet
import sys
sys.path.insert(0, f'{REPO_ROOT}/src')   # makes 'from model import AntennaGNN' work
print(f'Repo ready at {REPO_ROOT}')

In [ ]:
# Cell 3 â€” Mount Drive and set data paths & Section 1: Verify folder structure.
from google.colab import drive
drive.mount('/content/drive')
DATA_ROOT = '/content/drive/MyDrive/antenna_gnn'
RAW_DATA  = '/content/drive/MyDrive/antenna_dataset'

import os
for d in [f'{DATA_ROOT}/artifacts', f'{DATA_ROOT}/checkpoints',
          f'{DATA_ROOT}/figures',   f'{DATA_ROOT}/splits',
          f'{DATA_ROOT}/data/processed', f'{REPO_ROOT}/notebooks']:
    os.makedirs(d, exist_ok=True)
print(f'Drive mounted. DATA_ROOT={DATA_ROOT}')

import glob
RAW_PATHS = {
    25: f'{RAW_DATA}/training dataset/25x25/**/Mat_Files/*.mat',
    35: f'{RAW_DATA}/fine-tuning dataset/35x35/**/Mat_Files/*.mat',
    45: f'{RAW_DATA}/fine-tuning dataset/45x45/**/Mat_Files/*.mat',
    55: f'{RAW_DATA}/fine-tuning dataset/55x55/**/Mat_Files/*.mat',
}

for N, pattern in RAW_PATHS.items():
    files = sorted(glob.glob(pattern, recursive=True))
    count = len(files)
    print(f'Grid {N}x{N}: Found {count} files.')
    assert count > 0, f'No files found for {N}x{N}'
    print(f'  First: {files[0]}')
    print(f'  Last:  {files[-1]}')

In [ ]:
# Cell 4 â€” Section 2: Dataset summary table.
import random
import scipy.io as sio
import numpy as np

random.seed(42)
np.random.seed(42)

print(f'{"Grid":<10} | {"Total":<8} | {"Func. %":<10} | {"Metal Frac":<20} | {"Res. Freq (GHz)":<20}')
print('-'*76)

samples_dict = {}
for N, pattern in RAW_PATHS.items():
    files = sorted(glob.glob(pattern, recursive=True))
    total_count = len(files)
    
    sample_paths = random.sample(files, min(200, total_count))
    samples_dict[N] = sample_paths
    
    func_count = 0
    metal_fracs = []
    res_freqs = []
    
    for path in sample_paths:
        mat = sio.loadmat(path)
        pattern_arr = mat['patch_pattern']
        is_functioning = mat['resonant_freqs'].size > 0
        
        if is_functioning:
            func_count += 1
            res_freq = float(mat['resonant_freqs'].flatten()[0])
            res_freqs.append(res_freq)
            
        metal_frac = np.sum(pattern_arr) / (N**2)
        metal_fracs.append(metal_frac)
        
    func_pct = func_count / len(sample_paths) * 100
    mf_mean, mf_std = np.mean(metal_fracs), np.std(metal_fracs)
    rf_mean = np.mean(res_freqs) if res_freqs else 0.0
    rf_std = np.std(res_freqs) if res_freqs else 0.0
    
    print(f'{N}x{N:<7} | {total_count:<8} | {func_pct:>6.2f}%    | {mf_mean:>6.4f} +/- {mf_std:>6.4f} | {rf_mean:>6.4f} +/- {rf_std:>6.4f}')

In [ ]:
# Cell 5 â€” Section 3: Pattern visualization.
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os

sample_paths = samples_dict[25]

func_paths = []
non_func_paths = []
for path in sample_paths:
    mat = sio.loadmat(path)
    if mat['resonant_freqs'].size > 0:
        func_paths.append(path)
    else:
        non_func_paths.append(path)

selected_paths = (random.sample(func_paths, min(3, len(func_paths))) + 
                  random.sample(non_func_paths, min(3, len(non_func_paths))) + 
                  random.sample(sample_paths, min(3, len(sample_paths))))
selected_paths = selected_paths[:9]

fig, axes = plt.subplots(3, 3, figsize=(10, 10))
for ax, path in zip(axes.flatten(), selected_paths):
    mat = sio.loadmat(path)
    is_functioning = mat['resonant_freqs'].size > 0
    label = 'F' if is_functioning else 'NF'
    filename = os.path.splitext(os.path.basename(path))[0]
    
    ax.imshow(mat['patch_pattern'], cmap='binary')
    ax.set_title(f'{filename} ({label})')
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()
plt.savefig(f'{DATA_ROOT}/figures/pattern_grid.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 6 â€” Section 4: S11 spectrum visualization.
fig, ax = plt.subplots(figsize=(10, 6))
freq_axis = np.linspace(1.0, 4.0, 201)

func_plotted = False
non_func_plotted = False

for path in selected_paths:
    mat = sio.loadmat(path)
    s11 = mat['S11_dB'].flatten()
    is_functioning = mat['resonant_freqs'].size > 0
    
    if is_functioning:
        color = 'blue'
        label = 'Functioning' if not func_plotted else None
        func_plotted = True
    else:
        color = 'red'
        label = 'Non-functioning' if not non_func_plotted else None
        non_func_plotted = True
        
    ax.plot(freq_axis, s11, color=color, label=label)

ax.axhline(y=-10, color='black', linestyle='--', label='-10 dB threshold')
ax.set_xlabel('Frequency (GHz)')
ax.set_ylabel('S11 (dB)')
ax.legend()

plt.savefig(f'{DATA_ROOT}/figures/s11_spectra.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 7 — Section 5: Resonant frequency distribution.
import concurrent.futures
from tqdm.auto import tqdm

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

def load_res_freq(f):
    try:
        mat = sio.loadmat(f, variable_names=['resonant_freqs'])
        if mat['resonant_freqs'].size > 0:
            return float(mat['resonant_freqs'].flatten()[0])
    except:
        pass
    return None

for N, ax in zip([25, 35, 45, 55], axes.flatten()):
    files = sorted(glob.glob(RAW_PATHS[N], recursive=True))
    all_res_freqs = []
    
    print(f'Processing Grid {N}x{N} ({len(files)} files)...')
    # Use ThreadPoolExecutor to overcome Google Drive I/O latency
    with concurrent.futures.ThreadPoolExecutor(max_workers=32) as executor:
        results = list(tqdm(executor.map(load_res_freq, files), total=len(files)))
    
    all_res_freqs = [r for r in results if r is not None]
                
    ax.hist(all_res_freqs, bins=30, range=[1.0, 4.0], color='skyblue', edgecolor='black')
    ax.set_title(f'Grid {N}x{N} (n={len(all_res_freqs)})')
    ax.set_xlabel('Resonant Frequency (GHz)')
    ax.set_ylabel('Count')

plt.tight_layout()
plt.savefig(f'{DATA_ROOT}/figures/resonant_freq_distribution.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# Cell 8 — Section 6: Seed mask computation.
centroids = {}

def load_pattern(f):
    return sio.loadmat(f, variable_names=['patch_pattern'])['patch_pattern']

for N in [25, 35, 45, 55]:
    files = sorted(glob.glob(RAW_PATHS[N], recursive=True))
    BATCH = 5000
    always_metal = None
    
    print(f'Computing seed mask for {N}x{N} ({len(files)} files)...')
    for start in tqdm(range(0, len(files), BATCH)):
        batch_files = files[start:start+BATCH]
        
        with concurrent.futures.ThreadPoolExecutor(max_workers=32) as executor:
            batch_patterns = np.stack(list(executor.map(load_pattern, batch_files)))
            
        block = np.all(batch_patterns == 1, axis=0)
        always_metal = block if always_metal is None else (always_metal & block)
        
    np.save(f'{DATA_ROOT}/artifacts/seed_mask_{N}.npy', always_metal)
    
    coords = np.argwhere(always_metal)
    min_row, max_row = coords[:, 0].min(), coords[:, 0].max()
    min_col, max_col = coords[:, 1].min(), coords[:, 1].max()
    
    centroid = coords.mean(axis=0)
    centroid_mm = centroid * (32.375 / N)
    centroids[N] = {'pixel': centroid, 'mm': centroid_mm}
    
    print(f'Grid {N}x{N}:')
    print(f'  Bounding Box (row, col): min({min_row}, {min_col}) to max({max_row}, {max_col})')
    print(f'  Centroid (pixels):       ({centroid[0]:.2f}, {centroid[1]:.2f})')
    print(f'  Centroid (mm):           ({centroid_mm[0]:.3f}, {centroid_mm[1]:.3f})')
    print('-'*50)

print('Summary of Physical Centroids (mm):')
for N in [25, 35, 45, 55]:
    print(f'  {N}x{N}: ({centroids[N]["mm"][0]:.3f}, {centroids[N]["mm"][1]:.3f})')


In [ ]:
# Cell 9 â€” Completion checks.
import scipy.ndimage

print('Running completion checks...')

for N in [25, 35, 45, 55]:
    mask_path = f'{DATA_ROOT}/artifacts/seed_mask_{N}.npy'
    if os.path.exists(mask_path):
        print(f'PASS: {N}x{N} mask saved correctly.')
    else:
        print(f'FAIL: {N}x{N} mask not found at {mask_path}')

mask_25 = np.load(f'{DATA_ROOT}/artifacts/seed_mask_25.npy')
labeled_array, num_features = scipy.ndimage.label(mask_25)
if num_features > 0:
    sizes = scipy.ndimage.sum(mask_25, labeled_array, range(1, num_features + 1))
    max_size = np.max(sizes)
    if max_size > 50 and num_features == 1:
        print(f'PASS: 25x25 always-metal block is contiguous (size {max_size})')
    elif max_size > 50:
        print(f'PASS/WARN: 25x25 largest block is size {max_size}, but found {num_features} total blocks.')
    else:
        print(f'FAIL: 25x25 contiguous block too small (max size {max_size})')
else:
    print('FAIL: No always-metal pixels found in 25x25 mask.')